# Phoneme Regression: Neural Data → Phoneme Embeddings

This notebook regresses neural activity (`clean_data_binned`) onto phoneme embeddings
(PWESuite panphon, PWESuite token-IPA) using `BasicRegressor` with Kernel Ridge Regression.

### Pipeline
```
1. Load neural data (.mat)  →  2. Bin & clean trials/channels
                                        ↓
3. Lemmatise target labels  →  4. Load phoneme embeddings (PWESuite panphon & token-IPA)
                                        ↓
5. KernelRidge regression (BasicRegressor)  →  6. Plot R² over time
```

## 1. Imports

In [2]:
import gc
import numpy as np
import scipy.io as sio
import os
import sys
import collections
import pickle as pk
import mat73
import pandas as pd
from nltk.stem import WordNetLemmatizer
from sklearn.decomposition import PCA
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import Ridge
import matplotlib.pyplot as plt
from sklearn.kernel_approximation import Nystroem
from sklearn.pipeline import Pipeline


# Ensure we are at the project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, '.')

from utils.utils import reformat_raw, remove_number, plot_accuracy_plotly
from models.model import BasicRegressor

%load_ext autoreload
%autoreload 2

print('All imports OK')

All imports OK


## 2. Configuration

In [47]:
data_folder = 'data'
patient = 'VB'
all_patients = os.listdir(data_folder)
patient_folder_index = np.where([patient in s for s in all_patients])[0][0]
path = os.path.join(data_folder, all_patients[patient_folder_index], 'pictureNaming_all_data.mat')
print(path)

alpha = 0.05
bin_size = 100  # ms

data\2023-08-16 VB\pictureNaming_all_data.mat


## 3. Load Neural Data

In [48]:
try:
    all_data = np.array(mat73.loadmat(path)['all_data'], dtype=object)
except:
    all_data = sio.loadmat(path)['all_data']

# Remove broken trials (edge cases)
all_data = all_data[~np.array([isinstance(a, type(None)) for a in all_data[:, 0]])]
print(f'Loaded {len(all_data)} trials')

Loaded 203 trials


## 4. Assign Semantic Categories

In [49]:
df = pd.read_excel(os.path.join(data_folder, 'wordset picture naming expanded.xlsx'))
word_column = df.columns[0]
df.set_index(word_column, inplace=True)
df_filled = df.fillna(0).apply(pd.to_numeric)
category_series = df_filled.idxmax(axis=1).reset_index()
category_series.columns = [word_column, 'Category']
word_to_category_dict = dict(zip(category_series[word_column], category_series['Category']))

## 5. Process Neural Data: Bin & Clean

In [ ]:
fs = int(all_data[0, 10])
n_samples_per_bin = fs * bin_size // 1000


# Process target labels
target_label = reformat_raw(all_data[:, 6])
if 'Flashing' in path or 'auditoryNaming' in path or 'pictureNaming' in path:
    target_lexeme = np.array([remove_number(t) for t in target_label])
else:
    target_lexeme = target_label

# Assign and optionally merge semantic categories
word_category = [word_to_category_dict[word[:].lower()] for word in target_lexeme]
word_category = [w if w != 'fruit' and w != 'food (exclude fruit)' else 'food and fruit' for w in word_category]
all_data = np.concatenate([all_data, np.array(word_category)[:, np.newaxis]], axis=1)
print(collections.Counter(word_category))

# Optional: filter unwanted categories
unwanted_categories = []
category_mask = np.array([wc not in unwanted_categories for wc in word_category])
all_data = np.concatenate([all_data, category_mask[:, np.newaxis]], axis=1)
all_data = all_data[category_mask]
word_category = [wc for wc in word_category if wc not in unwanted_categories]


Counter({'abstract': 40, 'food and fruit': 37, 'animal': 36, 'body part': 32, 'nature': 31, 'objects and tools': 23, 'vehicle': 4})


C:\Users\Owner\AppData\Local\Temp\ipykernel_56536\2443050907.py:1: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)



In [59]:
# Extract timing and neural data
data = all_data[:, 0]
trial_onset = all_data[:, 1]
green_screen_onset = all_data[:, 2]
trial_offset = all_data[:, 3]
voice_onset = all_data[:, 4]
voice_offset = all_data[:, 5]
target_label = all_data[:, 6]
answer_label = all_data[:, 7]

# Load raw channel names if available (column 11 may be absent in some sessions).
# Guard: after category columns are appended, all_data[0, 11] could be a word-category
# string rather than a channel-name array — check it is NOT a plain string.
_col11 = all_data[0, 11] if all_data.shape[1] > 11 else None
_has_channel_names = (
    _col11 is not None
    and not isinstance(_col11, str)
    and np.asarray(_col11).size > 0
)
if _has_channel_names:
    raw_channel_names = np.array(all_data[0, 11]).squeeze()
    if raw_channel_names.shape[0] == data[0].shape[1] + 1:
        raw_channel_names = raw_channel_names[:-1]
    elif raw_channel_names.shape[0] != data[0].shape[1]:
        raise ValueError(
            f'Channel name count {raw_channel_names.shape[0]} does not match data channels {data[0].shape[1]}'
        )
else:
    # Fall back to numeric indices; name-based channel exclusion will be skipped
    raw_channel_names = np.array([f'ch{i}' for i in range(data[0].shape[1])])
    print('Warning: raw channel names not available — skipping name-based channel exclusion')

def _extract_bad_channel_indices(entries):
    parsed_indices = set()

    for entry in entries:
        if entry is None:
            continue

        flat_entry = np.asarray(entry, dtype=object).ravel()
        for channel_index in flat_entry:
            if channel_index is None:
                continue

            if isinstance(channel_index, str):
                channel_index = channel_index.strip()
                if not channel_index:
                    continue

            try:
                numeric_index = float(channel_index)
            except (TypeError, ValueError):
                continue

            if np.isnan(numeric_index):
                continue

            parsed_indices.add(int(numeric_index) - 1)

    return np.array(sorted(parsed_indices), dtype=int)

bad_trials = ~all_data[:, 9].astype(bool)
annotated_bad_channel_entries = all_data[:, 8]
bad_channels = _extract_bad_channel_indices(annotated_bad_channel_entries)
remaining_channel_indices = np.delete(np.arange(data[0].shape[1]), bad_channels)
channel_names = raw_channel_names[remaining_channel_indices]

print(f'Number of bad trials: {len(np.where(bad_trials == 0)[0])}')
print(f'Annotated bad channels (0-based): {bad_channels.tolist()}')

# Patient-specific channel selection
if patient == 'LH' and _has_channel_names:
    lh_excluded_filtered = np.array([
        index for index, channel_name in enumerate(channel_names)
        if str(channel_name).startswith(('V', 'O'))
    ], dtype=int)
    lh_excluded_channels = remaining_channel_indices[lh_excluded_filtered]
    bad_channels = np.union1d(bad_channels, lh_excluded_channels).astype(int)
    channel_names = np.delete(channel_names, lh_excluded_filtered, axis=0)
    print(f"Removing LH shanks V/O: {raw_channel_names[lh_excluded_channels].tolist()}")
elif patient == 'LH' and not _has_channel_names:
    print('LH: skipping shank-name-based exclusion (no channel names available)')

print(f'Total removed channels: {bad_channels.tolist()}')
print(f'Remaining channels: {len(channel_names)}')

# Bin the neural data
shortest_trial = min([d.shape[0] for d in data])
data = np.array([np.array([d for d in dt[:shortest_trial]]) for dt in data]).swapaxes(1, 2)
min_length = data.shape[2] // n_samples_per_bin * n_samples_per_bin
data = data[:, :, :min_length]
data_binned = data.reshape(data.shape[0], data.shape[1], -1, n_samples_per_bin).mean(axis=3)

# Free intermediate unbinned array
del data
gc.collect()

adjusted_fs = int(1000 / bin_size)

# Reformat timing cues
trial_onset = reformat_raw(trial_onset)
trial_offset = reformat_raw(trial_offset)
green_screen_onset = reformat_raw(green_screen_onset)
voice_onset = reformat_raw(voice_onset)
voice_offset = reformat_raw(voice_offset)

# Keep both forms: numbered labels for indexing and stripped lexemes for text embeddings
target_labels = np.array(reformat_raw(target_label))
target_lexeme = np.array([remove_number(t) for t in target_labels])
answer_label = reformat_raw(answer_label)

print(f'data_binned shape: {data_binned.shape}')


Number of bad trials: 0
Annotated bad channels (0-based): [15, 19, 21]
Total removed channels: [15, 19, 21]
Remaining channels: 47
data_binned shape: (203, 50, 58)


In [60]:
# Remove bad channels and bad trials
clean_data_binned = np.delete(data_binned, bad_channels, axis=1)
clean_data_binned = clean_data_binned[bad_trials]
clean_voice_onset = voice_onset[bad_trials]
clean_voice_offset = voice_offset[bad_trials]
clean_target_labels = np.squeeze(target_labels[bad_trials])
clean_target_lexeme = np.squeeze(target_lexeme[bad_trials])
clean_answer_label = answer_label[bad_trials]
clean_word_category = np.array(word_category)[bad_trials]
clean_channel_names = np.array(channel_names)

n_clean_channels = clean_data_binned.shape[1]
n_clean_trials = clean_data_binned.shape[0]

print(f'clean_data_binned shape: {clean_data_binned.shape}')
print(f'n_clean_trials={n_clean_trials}, n_clean_channels={n_clean_channels}')
print(f'clean_channel_names shape: {clean_channel_names.shape}')

clean_data_binned shape: (203, 47, 58)
n_clean_trials=203, n_clean_channels=47
clean_channel_names shape: (47,)


In [61]:
# Optional: select shanks of interest (sEEG patients)
# Uncomment and modify the lines below as needed

# shank_of_interest = ['T', 'J', 'S']
# shank_mask = np.array([str(cn)[0] in shank_of_interest for cn in clean_channel_names])
# clean_data_binned = clean_data_binned[:, shank_mask, :]
# clean_channel_names = clean_channel_names[shank_mask]
# n_clean_channels = clean_data_binned.shape[1]
# print(f'After shank selection: {clean_data_binned.shape}')

## 6. Lemmatise Target Words

In [62]:
lemmatizer = WordNetLemmatizer()

# target_lexeme: number-stripped labels (no trial suffix digits)
target_lexeme = np.array([str(w).lower() for w in clean_target_lexeme])

# target_lemma: lemmatised form for text embeddings
target_lemma = np.array([
    lemmatizer.lemmatize(''.join([c for c in a if c.isalpha()]), pos='n')
    for a in target_lexeme
])

print(f'Unique target labels: {len(np.unique(clean_target_labels))}')
print(f'Unique number-stripped lexemes: {len(np.unique(target_lexeme))}')
print(f'Unique lemmatised words: {len(np.unique(target_lemma))}')
print(f'Label examples: {clean_target_labels[:10]}')
print(f'Lexeme examples: {target_lexeme[:10]}')

Unique target labels: 55
Unique number-stripped lexemes: 53
Unique lemmatised words: 53
Label examples: ['nut' 'pear' 'moose' 'bear' 'bean' 'lime' 'newt' 'cat' 'fish' 'soup']
Lexeme examples: ['nut' 'pear' 'moose' 'bear' 'bean' 'lime' 'newt' 'cat' 'fish' 'soup']


## 7. Load Phoneme Embeddings

We load pre-computed PWESuite phoneme embeddings as regression targets:
- **PWESuite panphon** — RNN metric-learning model trained on panphon articulatory features
- **PWESuite token-IPA** — RNN metric-learning model trained on IPA token sequences

These embeddings capture phonological similarity between words and are generated
by `embeddings.py` using the PWESuite framework.

In [63]:
image_folder = os.path.join(data_folder, 'pictureNaming extended all')
embeddings_folder = os.path.join('embeddings', os.path.basename(image_folder))

def _normalize_token(token):
    token = str(token).strip().lower()
    token = remove_number(token)
    token = ''.join(ch for ch in token if ch.isalpha())
    return token

def _normalize_tokens(tokens):
    return np.array([_normalize_token(t) for t in tokens])

def _map_to_target(embed_array, embed_words, target_labels, alt_label_sets=None):
    """Return [N_trials, D] array aligned to target_labels with strict matching."""
    words_norm = _normalize_tokens(embed_words)
    lookup = {}
    for i, w in enumerate(words_norm):
        if w and w not in lookup:
            lookup[w] = i

    out = []
    missing = []
    alt_label_sets = alt_label_sets or []

    for i, t_raw in enumerate(target_labels):
        candidates = [t_raw] + [labels[i] for labels in alt_label_sets]
        matched_index = None
        tried = []

        for cand in candidates:
            cand_norm = _normalize_token(cand)
            if not cand_norm:
                continue
            tried.append(cand_norm)
            if cand_norm in lookup:
                matched_index = lookup[cand_norm]
                break

        if matched_index is None:
            missing.append((t_raw, tried))
            continue

        out.append(np.asarray(embed_array[matched_index]).squeeze())

    if missing:
        preview = '; '.join([f"{raw} -> {tries}" for raw, tries in missing[:10]])
        raise ValueError(
            f"No embedding match for {len(missing)}/{len(target_labels)} labels. "
            f"Examples: {preview}"
        )

    return np.asarray(out, dtype=np.float32)


# -- PWESuite panphon embeddings ----------------------------------------------
panphon_pkl = os.path.join(embeddings_folder, 'pwesuite_panphon_embeddings.pk')
with open(panphon_pkl, 'rb') as f:
    panphon_results = pk.load(f)

panphon_words = np.array(panphon_results['words'])
target_word_embed_panphon = _map_to_target(
    np.array(panphon_results['phoneme_embedding'], dtype=np.float32),
    panphon_words,
    clean_target_labels,
    alt_label_sets=[clean_target_lexeme, target_lemma],
)
del panphon_results
print(f'PWESuite panphon : {target_word_embed_panphon.shape}')


# -- PWESuite token-IPA embeddings --------------------------------------------
token_ipa_pkl = os.path.join(embeddings_folder, 'pwesuite_token_ipa_embeddings.pk')
with open(token_ipa_pkl, 'rb') as f:
    token_ipa_results = pk.load(f)

token_ipa_words = np.array(token_ipa_results['words'])
target_word_embed_token_ipa = _map_to_target(
    np.array(token_ipa_results['phoneme_embedding'], dtype=np.float32),
    token_ipa_words,
    clean_target_labels,
    alt_label_sets=[clean_target_lexeme, target_lemma],
)
del token_ipa_results
gc.collect()
print(f'PWESuite token-IPA : {target_word_embed_token_ipa.shape}')

PWESuite panphon : (203, 300)
PWESuite token-IPA : (203, 300)


## 8. Regression: Neural Activity → Phoneme Embeddings

For each embedding type we fit a `BasicRegressor` (KernelRidge with RBF kernel)
that maps `clean_data_binned` → embedding, sliding across time bins.

In [64]:
# ── Regression parameters ────────────────────────────────────────
n_bins_history = 10
n_epochs = 50
y_pca_components = 50
krr_alpha = 1.5

In [65]:
# ── PWESuite panphon regression ───────────────────────────────────
y_reducer = PCA(y_pca_components)
nystroem = Nystroem(kernel='rbf')
regressor = Pipeline([
    ('nystroem', nystroem),
    ('ridge', Ridge(alpha=krr_alpha))
])

br_panphon = BasicRegressor(regressor, y_reducer=y_reducer)
br_panphon.load_data(clean_data_binned.swapaxes(1, 2), target_word_embed_panphon, n_bins_history=n_bins_history, labels=clean_target_lexeme)
br_panphon.fit(n_epochs=n_epochs, parallel=10, compute_retrieval=True)
print('PWESuite panphon regression done')

d:\OneDrive - Northwestern University\PycharmProjects\Speech\main\.\models\model.py:367: RuntimeWarning:

Parallel regression failed due to a notebook multiprocessing pickling issue; falling back to sequential execution.



PWESuite panphon regression done


In [66]:
# ── PWESuite token-IPA regression ────────────────────────────────
y_reducer = PCA(y_pca_components)
nystroem = Nystroem(kernel='rbf')
regressor = Pipeline([
    ('nystroem', nystroem),
    ('ridge', Ridge(alpha=krr_alpha))
])

br_token_ipa = BasicRegressor(regressor, y_reducer=y_reducer)
br_token_ipa.load_data(clean_data_binned.swapaxes(1, 2), target_word_embed_token_ipa, n_bins_history=n_bins_history, labels=clean_target_lexeme)
br_token_ipa.fit(n_epochs=n_epochs, parallel=10, compute_retrieval=True)
print('PWESuite token-IPA regression done')

d:\OneDrive - Northwestern University\PycharmProjects\Speech\main\.\models\model.py:367: RuntimeWarning:

Parallel regression failed due to a notebook multiprocessing pickling issue; falling back to sequential execution.



PWESuite token-IPA regression done


## 9. Plot Regression Results (R² over Time)

In [67]:
fig, ax = plot_accuracy_plotly(
    br_panphon.all_test_score.mean(0),
    #br_token_ipa.all_test_score.mean(0),
    br_panphon.all_chance.mean(0),
    data_std=[
        br_panphon.all_test_score.std(0),
        #br_token_ipa.all_test_score.std(0), --- IGNORE ---
        br_panphon.all_chance.std(0),
    ],
    lines=[
        0 - np.nanmean(trial_onset),
        np.nanmean(green_screen_onset) - np.nanmean(trial_onset),
        np.nanmean(clean_voice_onset) - np.nanmean(trial_onset),
        np.nanmean(clean_voice_offset) - np.nanmean(trial_onset),
    ],
    line_labels=['trial onset', 'go cue', 'voice on', 'voice off'],
    data_labels=['PWESuite panphon', 'PWESuite token-IPA', 'chance'],
    back=np.nanmean(trial_onset),
    forward=clean_data_binned.shape[2] / adjusted_fs - np.nanmean(trial_onset),
    ylabel='1-SSE/SST (R²)',
    tick_interval=1,
    title='Aligned to Trial Onset: Regression to Phoneme Embeddings',
)

fig.show()

## 9b. Plot Retrieval Accuracy (Top-1)

In [ ]:
# ── Retrieval: Top-1 accuracy ────────────────────────────────────
fig_ret1, _ = plot_accuracy_plotly(
    br_panphon.all_retrieval_top1.mean(0),
    #br_token_ipa.all_retrieval_top1.mean(0),
    br_panphon.all_retrieval_chance_top1.mean(0),
    #br_token_ipa.all_retrieval_chance_top1.mean(0),
    lines=[
        0 - np.nanmean(trial_onset),
        np.nanmean(green_screen_onset) - np.nanmean(trial_onset),
        np.nanmean(clean_voice_onset) - np.nanmean(trial_onset),
        np.nanmean(clean_voice_offset) - np.nanmean(trial_onset),
    ],
    data_std=[
        br_panphon.all_retrieval_top1.std(0),
        #br_token_ipa.all_retrieval_top1.std(0),
        0
        #br_token_ipa.all_retrieval_chance_top1.std(0),
    ],
    line_labels=['trial onset', 'go cue', 'voice on', 'voice off'],
    data_labels=[
        'PWESuite panphon',
        #'PWESuite token-IPA',
        'shuffled chance',
        #'token-IPA shuffled chance',
    ],
    back=np.nanmean(trial_onset),
    forward=clean_data_binned.shape[2] / adjusted_fs - np.nanmean(trial_onset),
    ylabel='Top-1 Retrieval Accuracy',
    tick_interval=1,
    title='Retrieval Top-1 Accuracy',
)
fig_ret1.show()

: 

## 10. Save Results

In [ ]:
save_base = os.path.join('results', patient, 'phoneme_regression')
os.makedirs(save_base, exist_ok=True)

with open(os.path.join(save_base, 'panphon_r2_curves.pk'), 'wb') as f:
    pk.dump(br_panphon.all_test_score, f)

with open(os.path.join(save_base, 'token_ipa_r2_curves.pk'), 'wb') as f:
    pk.dump(br_token_ipa.all_test_score, f)

with open(os.path.join(save_base, 'panphon_retrieval.pk'), 'wb') as f:
    pk.dump({
        'top1': br_panphon.all_retrieval_top1,
        'chance_top1': br_panphon.all_retrieval_chance_top1,
    }, f)

with open(os.path.join(save_base, 'token_ipa_retrieval.pk'), 'wb') as f:
    pk.dump({
        'top1': br_token_ipa.all_retrieval_top1,
        'chance_top1': br_token_ipa.all_retrieval_chance_top1,
    }, f)

print(f'Results saved to {save_base}')

## 11. Load & Re-plot Saved Results (optional)

In [ ]:
load_base = os.path.join('results', patient, 'phoneme_regression')

with open(os.path.join(load_base, 'panphon_r2_curves.pk'), 'rb') as f:
    saved_panphon_scores = pk.load(f)

with open(os.path.join(load_base, 'token_ipa_r2_curves.pk'), 'rb') as f:
    saved_token_ipa_scores = pk.load(f)

print(f'Loaded panphon  scores: {saved_panphon_scores.shape}')
print(f'Loaded token-IPA scores: {saved_token_ipa_scores.shape}')

fig2, _ = plot_accuracy_plotly(
    saved_panphon_scores.mean(0),
    saved_token_ipa_scores.mean(0),
    lines=[
        0 - np.nanmean(trial_onset),
        np.nanmean(green_screen_onset) - np.nanmean(trial_onset),
        np.nanmean(clean_voice_onset) - np.nanmean(trial_onset),
        np.nanmean(clean_voice_offset) - np.nanmean(trial_onset),
    ],
    line_labels=['trial onset', 'go cue', 'voice on', 'voice off'],
    data_labels=['PWESuite panphon', 'PWESuite token-IPA'],
    back=np.nanmean(trial_onset),
    forward=clean_data_binned.shape[2] / adjusted_fs - np.nanmean(trial_onset),
    ylabel='1-SSE/SST (R²)',
    tick_interval=1,
    title='Aligned to Trial Onset: Regression to Phoneme Embeddings (loaded)',
)
fig2.show()